# Ettin social adaptation: backbone → Matryoshka prefixes → transformer exits
One self-contained A100 notebook: all implementation is inline, with resumable Google Drive checkpoints.
Default initialization is the selected backbone from `ettin-pilot-01` (not the old exit heads).
The inspected Drive listing has completed pilot checkpoints, but no completed diverse-backbone checkpoint or export.
Run all cells in order. Each stage restores its validation-best weights before the next stage starts.

**Training:** MS MARCO + NQ + SQuAD + AllNLI retrieval; GoEmotions multilabel emotion classification;
TweetTopic temporal topic classification. Twitter URL original training data are supported via a local Drive file;
the automatically downloaded MTEB Twitter URL **test set is evaluation-only**, never repurposed for training.
If the training file is absent, that limitation is printed and recorded, and the other tasks run normally.

Defaults: 10,000 backbone updates at full width, 5,000 prefix updates at 128/256/384D, and 10,000 frozen-backbone
transformer-exit updates. These are editable budgets, not a claim more updates always help. No tiny long-passage
pool oversampling: this run targets short social text at a 256-token training cap. Actual early exits execute
only the requested backbone prefix plus its one transformer branch. Full-depth inference skips the branches.

**The final cell unassigns the Colab runtime.** All exports, reports and reload checks run before it.
This is the agreed research continuation; data terms and unresolved licensing details remain in the provenance record.


In [26]:
# Install into this notebook's kernel. Keep Colab's CUDA-enabled PyTorch.
# Restart the kernel after this cell if it already imported different versions.
%pip install -q transformers==5.1.0 datasets==4.5.0 accelerate==1.12.0 sentence-transformers==5.2.2 safetensors
%pip install -q scikit-learn==1.7.2 matplotlib


In [27]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = '/content/hf-cache' if os.path.isdir('/content') else '/tmp/encoder-hf-cache'
import copy, gc, hashlib, json, math, random, shutil, time, uuid
from pathlib import Path
from collections import defaultdict
from contextlib import nullcontext
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file, save_file
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')
DEVICE = 'cuda'
assert torch.cuda.is_available(), 'Select an A100 GPU runtime in Colab.'
assert torch.cuda.is_bf16_supported(), 'This notebook uses BF16 on a supported GPU.'
print(torch.cuda.get_device_name(), 'PyTorch:', torch.__version__)
torch.set_float32_matmul_precision('high')
import csv, gzip, re
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, average_precision_score
from transformers.models.modernbert.modeling_modernbert import create_bidirectional_mask, create_bidirectional_sliding_window_mask
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NVIDIA A100-SXM4-40GB PyTorch: 2.11.0+cu128


## Settings
Use a new run name when changing settings. The 10,000-step default applies to new runs; existing 3,000-step checkpoints retain their original contract and cosine schedule. Editing PHASES does not extend an already executing loop or migrate its checkpoints. `SOURCE_RUN` may point to the previous diverse backbone run or pilot.
Set `INIT_MODE='pretrained'` to run without an existing project checkpoint. The entire pipeline is contained here.
`TWITTER_TRAIN_FILE` accepts the original tab-separated format (sentence1, sentence2, `(votes,6)`, URL),
or JSONL with `sentence1`, `sentence2`, binary `label`, and `url`. Get the training split from the authors:
https://github.com/lanwuwei/Twitter-URL-Corpus . Do not supply MTEB's test set here.

For a smoke run, use 2 updates per stage, batch sizes 4, train caps 64, validation/test caps 8,
evaluation/checkpoint intervals 1, and exit depths `[1,5,9]`. Select `RUN_NAME` before running the remaining cells.


In [28]:
RUN_NAME = 'ettin-social-unified-02'
INIT_MODE = 'research_continuation'  # user-selected; or 'pretrained' for a separate initialization
SOURCE_RUN = Path('/content/drive/MyDrive/encoder_experiments/ettin-pilot-01')
SOURCE_STAGE = 'student'
SOURCE_CHOICE = 'best'
ROOT = Path('/content/drive/MyDrive/encoder_experiments') / RUN_NAME
LOCAL = Path('/content/encoder-work') / RUN_NAME
assert ROOT.resolve()!=SOURCE_RUN.resolve()
ROOT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)
CFG = dict(experiment='ettin', recipe='social-unified-v1', seed=44, ettin_size='32m',
           short_length=256, max_length=512, teacher_batch_size=16, eval_batch_size=32,
           temperature=.05, alignment_weight=.25, relational_weight=.2,
           false_negative_margin=.05, max_full_mrr_drop=.03, aux_weight=.3,
           gradient_checkpointing=True, eval_every=250, checkpoint_every=100, log_every=25,
           validation_rows=128, test_rows=256, shuffle_buffer=10000, scan_limit=800000,
           max_text_chars=24000, mining_pool=8192, mining_candidates=8)
DIMS=[128,256,384]
GTE=dict(id='Alibaba-NLP/gte-modernbert-base',revision='e7f32e3c00f91d699e8c43b53106206bcc72bb22')
ETTIN=dict(id='jhu-clsp/ettin-encoder-32m',revision='1b8ba06455dd44f80fc9c1ca9e22806157a57379')
DATA_SPECS={
 'nq':dict(id='sentence-transformers/natural-questions',revision='f9e894e1081e206e577b4eaa9ee6de2b06ae6f17',subset='pair',split='train'),
 'squad':dict(id='sentence-transformers/squad',revision='d84c8c2ef64693264c890bb242d2e73fc0a46c40',subset='pair',split='train'),
 'all_nli':dict(id='sentence-transformers/all-nli',revision='d482672c8e74ce18da116f430137434ba2e52fab',subset='triplet',split='train'),
}
NANO=dict(id='sentence-transformers/NanoBEIR-en',revision='beb106fbcfaa599c508c667041bf8c85fd78736b')
BENCHMARK_NAMES=['NanoNQ','NanoHotpotQA','NanoMSMARCO']
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(CFG['seed'])
def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
def text_id(s):
    return hashlib.sha256(' '.join(s.split()).casefold().encode()).hexdigest()
def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp-' + uuid.uuid4().hex)
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False))
    os.replace(tmp, path)
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()



# Independent optimizers/schedules; each starts from the previous stage's selected model.
PHASES = {
    'backbone': dict(steps=10000, lr=1e-5, batch=48, dims=[384]),
    'prefix': dict(steps=5000, lr=5e-6, batch=48, dims=DIMS),
    'exits': dict(steps=10000, lr=3e-5, batch=24, dims=DIMS),
}
EXIT_DEPTHS = None  # every layer except full depth; e.g. [1,3,5,7,9] reduces stored branches
TWITTER_TRAIN_FILE = None  # e.g. '/content/drive/MyDrive/datasets/Twitter_URL_Corpus_train.txt'
TWITTER_TRAIN_SHA = file_sha(TWITTER_TRAIN_FILE) if TWITTER_TRAIN_FILE else None
CFG.update(recipe='social-unified-v1', short_length=256, max_length=512, batch_size=48,
           eval_batch_size=32, eval_every=250, checkpoint_every=100, validation_rows=128,
           test_rows=256, aux_weight=.3, max_full_mrr_drop=.03)
EVAL_LENGTHS=[256]
SOURCE_CAPS=dict(nq=20000,squad=20000,all_nli=30000,msmarco=60000,go_emotions=30000,tweet_topic=10000)
DATA_SPECS.update({
 'msmarco':dict(id='bclavie/msmarco-500k-triplets',revision='cb1a85c1261fa7c65f4ea43f94e50f8b467c372f',subset='default',split='train'),
 'go_emotions':dict(id='google-research-datasets/go_emotions',revision='add492243ff905527e67aeb8b80c082af02207c3',subset='simplified'),
 'tweet_topic':dict(id='cardiffnlp/tweet_topic_single',revision='87b7a0d1c402dbb481db649569c556d9aa27ac05'),
})
TWITTER_TEST=dict(id='mteb/twitterurlcorpus-pairclassification',revision='ee5d15331196ec9f72e8393779b20098c10e29f6')
MIX_CYCLE=['msmarco','go_emotions','nq','tweet_topic','msmarco','all_nli','squad','go_emotions','tweet_topic']
if TWITTER_TRAIN_FILE:
    SOURCE_CAPS['twitter_url']=30000
    MIX_CYCLE += ['twitter_url','twitter_url']
else:
    print('Twitter URL: evaluation-only; set TWITTER_TRAIN_FILE to enable its original training split.')
TASKS={n:('emotion' if n=='go_emotions' else 'topic' if n=='tweet_topic' else 'paraphrase' if n=='twitter_url' else 'retrieval') for n in SOURCE_CAPS}
assert set(MIX_CYCLE)==set(SOURCE_CAPS)
assert all(p['steps']>0 and 1<p['batch']<=min(SOURCE_CAPS.values()) for p in PHASES.values())
assert INIT_MODE in ('pretrained','research_continuation')
assert ROOT.resolve()!=SOURCE_RUN.resolve()
print('Stages:',PHASES,'Destination:',ROOT)


Twitter URL: evaluation-only; set TWITTER_TRAIN_FILE to enable its original training split.
Stages: {'backbone': {'steps': 10000, 'lr': 1e-05, 'batch': 48, 'dims': [384]}, 'prefix': {'steps': 5000, 'lr': 5e-06, 'batch': 48, 'dims': [128, 256, 384]}, 'exits': {'steps': 10000, 'lr': 3e-05, 'batch': 24, 'dims': [128, 256, 384]}} Destination: /content/drive/MyDrive/encoder_experiments/ettin-social-unified-02


## Data provenance and terms
This run explicitly includes [MS MARCO research-use data](https://microsoft.github.io/msmarco/).
The third-party 500k-triplet mirror does not replace those upstream terms.
[Twitter URL](https://github.com/lanwuwei/Twitter-URL-Corpus) is CC BY-NC-SA 3.0 plus Twitter terms;
its MTEB mirror labels the license unknown, so retain the original terms too.
[GoEmotions](https://huggingface.co/datasets/google-research-datasets/go_emotions) lists Apache-2.0 via its repository;
that does not separately establish rights to every Reddit comment.
[TweetTopic](https://huggingface.co/datasets/cardiffnlp/tweet_topic_single) lists `other`: underlying Twitter rights
are unresolved here. Inclusion follows the user's research choice, not a declaration of commercial clearance.
NQ (CC BY-SA 3.0), SQuAD (CC BY-SA 4.0), and mixed SNLI/MultiNLI terms remain recorded.
Model licensing is separate from dataset access/use and underlying content rights. No blanket MIT/Apache claim
is assigned to the resulting weights. Cards and model notices are saved with the export.


In [29]:
LICENSE_RECORD=dict(review_date='2026-09-24',models={'ettin':'MIT','gte':'Apache-2.0'},
 datasets={
  'squad':dict(terms=['CC-BY-SA-4.0'],upstream=['https://rajpurkar.github.io/SQuAD-explorer/']),
  'nq':dict(terms=['CC-BY-SA-3.0'],upstream=['https://huggingface.co/datasets/google-research-datasets/natural_questions','https://github.com/google-research-datasets/natural-questions']),
  'all_nli':dict(terms=['SNLI: CC-BY-SA-4.0','MultiNLI: OANC terms; CC-BY-3.0; CC-BY-SA-3.0; US public domain'],
                 upstream=['https://nlp.stanford.edu/projects/snli/','https://aclanthology.org/N18-1101.pdf']),
 },benchmarks={'NanoNQ':'Natural Questions: CC-BY-SA-3.0','NanoHotpotQA':'CC-BY-SA-4.0'},
 excluded=['MS MARCO','GooAQ','wholesale NVIDIA embed-nemotron-dataset-v1','Stack Exchange wrappers','S2ORC'],
 transformations='Normalize whitespace for identities; group/split/filter; mine lexical training negatives; tokenize/truncate at runtime.',
 obligations='Retain attribution, source links, changes and applicable ShareAlike notices when redistributing adapted data. No blanket license assigned to derived weights.',
 lineage='Old MS MARCO-trained pilot retained' if INIT_MODE=='research_continuation' else 'Pinned pretrained Ettin; no project MS MARCO pilot loaded')
LICENSE_RECORD['datasets'].update({
 'msmarco':dict(terms=['MS MARCO non-commercial research terms; third-party mirror'],upstream=['https://microsoft.github.io/msmarco/']),
 'go_emotions':dict(terms=['Apache-2.0 listed via repository; underlying comment rights not independently cleared'],upstream=['https://github.com/google-research/google-research/tree/master/goemotions']),
 'tweet_topic':dict(terms=['Other/custom; underlying Twitter terms unresolved'],upstream=['https://huggingface.co/datasets/cardiffnlp/tweet_topic_single']),
 'twitter_url':dict(terms=['CC-BY-NC-SA-3.0; Twitter terms; MTEB mirror license unknown'],upstream=['https://github.com/lanwuwei/Twitter-URL-Corpus'],training_enabled=bool(TWITTER_TRAIN_FILE)),
})
LICENSE_RECORD['excluded']=['GooAQ','Reddit title-body','wholesale NVIDIA collection','S2ORC']
LICENSE_RECORD['lineage']='Research continuation including MS MARCO' if INIT_MODE=='research_continuation' else 'Pretrained Ettin followed by research data including MS MARCO'
LICENSE_RECORD['benchmarks']['NanoMSMARCO']='MS MARCO research-use terms'
source_info=dict(mode=INIT_MODE,model=ETTIN)
source_state=None
if INIT_MODE=='research_continuation':
    source_config_path=SOURCE_RUN/'config.json'
    if not source_config_path.is_file():
        raise FileNotFoundError(f'Source configuration missing: {source_config_path}. Check SOURCE_RUN and the Drive mount.')
    source_contract=json.loads(source_config_path.read_text())
    if source_contract.get('ettin')!=ETTIN or source_contract.get('gte')!=GTE:
        raise ValueError('Source Ettin/GTE model revisions do not match this notebook. Check SOURCE_RUN; do not bypass model compatibility checks.')
    expected_source_contract=digest(source_contract)
    def read_source(path):
        path=Path(path)
        meta=json.loads((path/'COMMITTED.json').read_text())
        if meta.get('contract')!=expected_source_contract:
            raise ValueError(f"config.json hash mismatch: expected {expected_source_contract}, manifest contains {meta.get('contract')}. The source config may have changed after training, or these files belong to different runs.")
        local=LOCAL/'source.pt'
        shutil.copyfile(path/'state.pt',local)
        actual_sha=file_sha(local)
        if actual_sha!=meta.get('sha256'):
            raise ValueError(f"state.pt checksum mismatch: expected {meta.get('sha256')}, copied file is {actual_sha}. The Drive file may be incomplete or modified.")
        state=torch.load(local,map_location='cpu',weights_only=False)  # only your own checkpoints
        if state.get('contract')!=meta['contract']:
            raise ValueError('Checkpoint payload contract differs from its COMMITTED.json manifest.')
        if state.get('step')!=meta.get('step'):
            raise ValueError(f"Checkpoint step {state.get('step')} differs from manifest step {meta.get('step')}.")
        if state.get('family')!='ettin' or state.get('stage')!=SOURCE_STAGE:
            raise ValueError(f"Expected an Ettin backbone checkpoint: family='ettin', stage={SOURCE_STAGE!r}; got family={state.get('family')!r}, stage={state.get('stage')!r}. Exit-head checkpoints cannot initialize the backbone.")
        missing={'model','spec','best_step','data_id'}-state.keys()
        if missing:raise ValueError(f'Missing checkpoint fields: {sorted(missing)}')
        return state,meta

    def select_source():
        lock=ROOT/'source.json'
        if lock.exists():
            info=json.loads(lock.read_text())
            wanted=dict(mode=INIT_MODE,run=str(SOURCE_RUN),choice=SOURCE_CHOICE,stage=SOURCE_STAGE)
            if any(info.get(k)!=v for k,v in wanted.items()):
                raise ValueError(f'Source settings differ from the existing lock {lock}. Use the original settings or a new RUN_NAME.')
            state,meta=read_source(Path(info['checkpoint']))
            if meta['sha256']!=info['sha256']:
                raise ValueError(f'Locked source checksum changed: {lock}')
            return info,state
        parent=SOURCE_RUN/'checkpoints'/SOURCE_STAGE
        candidates=sorted((p for p in parent.iterdir() if p.is_dir() and not p.name.startswith('.') and (p/'COMMITTED.json').is_file()),reverse=True) if parent.is_dir() else []
        print('Source checkpoint directory:',parent,'| committed generations:',len(candidates),flush=True)
        if not candidates:
            stages=SOURCE_RUN/'checkpoints'
            available=sorted(p.name for p in stages.iterdir() if p.is_dir()) if stages.is_dir() else []
            raise FileNotFoundError(f'No committed checkpoints in {parent}. Available stages: {available}. Check SOURCE_RUN/SOURCE_STAGE and that state.pt plus COMMITTED.json were copied to Drive.')
        failures=[]
        def attempt(path):
            try:return read_source(path)
            except (OSError,ValueError,AssertionError,EOFError,RuntimeError,KeyError) as exc:
                reason=f'{path.name}: {type(exc).__name__}: {str(exc) or "assertion without a message"}'
                failures.append(reason);print('Rejected source',reason,flush=True)
                return None
        latest=None
        for path in candidates:
            loaded=attempt(path)
            if loaded is not None:
                latest,meta=loaded;break
        if latest is None:
            raise RuntimeError('No valid source checkpoint. Nothing was loaded and verification was not bypassed.\n'+'\n'.join(failures))
        selected=latest['best_step'] if SOURCE_CHOICE=='best' else latest['step'] if SOURCE_CHOICE=='latest' else int(SOURCE_CHOICE)
        if latest['step']!=selected:
            # Try every generation of the selected step: one interrupted copy must not mask a valid copy.
            latest=None
            for candidate in candidates:
                if not candidate.name.startswith(f'{selected:08d}-'):continue
                loaded=attempt(candidate)
                if loaded is not None and loaded[0]['step']==selected:
                    latest,meta=loaded;path=candidate;break
            if latest is None:
                raise RuntimeError(f'Selected source step {selected} is missing or invalid in {parent}. Restore that checkpoint; the loader will not silently substitute another step.\n'+'\n'.join(failures))
        info=dict(mode=INIT_MODE,run=str(SOURCE_RUN),stage=SOURCE_STAGE,choice=SOURCE_CHOICE,data_id=latest['data_id'],
                  step=selected,checkpoint=str(path),sha256=meta['sha256'],model=ETTIN)
        write_json(lock,info)
        return info,latest
    source_info,source_state=select_source()
    print('Verified source checkpoint:',source_info['checkpoint'],'| selected step:',source_info['step'],flush=True)

contract=dict(config=CFG,ettin=ETTIN,gte=GTE,data=DATA_SPECS,nano=NANO,source=source_info,
              source_caps=SOURCE_CAPS,mix_cycle=MIX_CYCLE,dimensions=DIMS,eval_lengths=EVAL_LENGTHS,
              benchmark_names=BENCHMARK_NAMES,licenses=LICENSE_RECORD,phases=PHASES,exit_depths=EXIT_DEPTHS,
              twitter_test=TWITTER_TEST,twitter_train_sha=TWITTER_TRAIN_SHA,tasks=TASKS,implementation=1)
CONTRACT_ID=digest(contract)
if (ROOT/'config.json').exists():assert json.loads((ROOT/'config.json').read_text())==contract, 'Settings changed: choose a new RUN_NAME.'
else:write_json(ROOT/'config.json',contract)
write_json(ROOT/'LICENSE_PROVENANCE.json',LICENSE_RECORD)
# Save exact pinned model/dataset cards and available license files, not just mutable URLs.
NOTICE_DIR=ROOT/'notices';NOTICE_DIR.mkdir(exist_ok=True)
for name,spec,kind in [('ettin',ETTIN,'model'),('gte',GTE,'model')]+[(k,v,'dataset') for k,v in DATA_SPECS.items()]+[('nano',NANO,'dataset'),('twitter-test',TWITTER_TEST,'dataset')]:
    card=hf_hub_download(spec['id'],'README.md',revision=spec['revision'],repo_type=kind)
    shutil.copyfile(card,NOTICE_DIR/f'{name}-README.md')
# Hugging Face's typed missing-file exception lets network/auth failures remain visible.
from huggingface_hub.errors import EntryNotFoundError
for name,spec in [('ettin',ETTIN),('gte',GTE)]:
    for filename in ['LICENSE','LICENSE.txt']:
        try:lic=hf_hub_download(spec['id'],filename,revision=spec['revision'])
        except EntryNotFoundError:continue
        shutil.copyfile(lic,NOTICE_DIR/f'{name}-{filename}')
print('Recorded license provenance and initialization:',source_info)
EMBEDDED_MODEL_NOTICES = {'ettin-MIT.txt': {'text': 'MIT License\n\nCopyright (c) 2025 Johns Hopkins University Center for Language and Speech Processing\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE. ', 'url': 'https://raw.githubusercontent.com/JHU-CLSP/ettin-encoder-vs-decoder/6cbf954a2ecaa7c753be6468a1bccdef401d5f93/LICENSE'}, 'Apache-2.0.txt': {'text': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'url': 'https://www.apache.org/licenses/LICENSE-2.0.txt'}}
for name,notice in EMBEDDED_MODEL_NOTICES.items():
    (NOTICE_DIR/name).write_text(notice['text'])
write_json(NOTICE_DIR/'model-license-sources.json',{k:v['url'] for k,v in EMBEDDED_MODEL_NOTICES.items()})


Source checkpoint directory: /content/drive/MyDrive/encoder_experiments/ettin-pilot-01/checkpoints/student | committed generations: 3
Verified source checkpoint: /content/drive/MyDrive/encoder_experiments/ettin-pilot-01/checkpoints/student/00000400-88ddf87f | selected step: 400
Recorded license provenance and initialization: {'mode': 'research_continuation', 'run': '/content/drive/MyDrive/encoder_experiments/ettin-pilot-01', 'stage': 'student', 'choice': 'best', 'data_id': '808e89183ca1c2c9aaaa5a91808723ed41e474c94f1fa1f5bc95556168c3ab3e', 'step': 400, 'checkpoint': '/content/drive/MyDrive/encoder_experiments/ettin-pilot-01/checkpoints/student/00000400-88ddf87f', 'sha256': 'baf59fec1759927a51812ce763c61aa3a4d808d21cc84e6330c120b36e2d0cea', 'model': {'id': 'jhu-clsp/ettin-encoder-32m', 'revision': '1b8ba06455dd44f80fc9c1ca9e22806157a57379'}}


## Shared encoder and external teacher


In [30]:
def mean_pool(hidden, mask):
    mask = mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden.float()*mask).sum(1) / mask.sum(1).clamp_min(1)

# Input adapters belong to the model, so evaluation, training and export use the same tokenizer/pooling.
class EmbeddingModel(nn.Module):
    def __init__(self, backbone, tokenizer, output_dim, family, spec):
        super().__init__()
        self.backbone, self.tokenizer = backbone, tokenizer
        self.output_dim, self.family, self.spec = output_dim, family, copy.deepcopy(spec)
        self.max_length = CFG['max_length']
        # Ettin's alignment head is training-only. Retrieval always uses its native 384/512 dimensions.
        self.align = nn.Identity() if output_dim == 768 else nn.Linear(output_dim,768,bias=False)
    def forward(self, texts):
        tokens = self.tokenizer(texts, padding=True, truncation=True, max_length=self.max_length,
                                return_tensors='pt').to(next(self.parameters()).device)
        if self.family == 'neobert':
            h = self.backbone(tokens['input_ids'], tokens['attention_mask'])
        else:
            h = self.backbone(**tokens).last_hidden_state
        return mean_pool(h, tokens['attention_mask'])

def count_parameters(model):
    return dict(total=sum(p.numel() for p in model.parameters()),
                backbone=sum(p.numel() for p in model.backbone.parameters()),
                training_only_alignment=sum(p.numel() for p in model.align.parameters()))

def fresh_ettin():
    tok = AutoTokenizer.from_pretrained(ETTIN['id'],revision=ETTIN['revision'])
    backbone = AutoModel.from_pretrained(ETTIN['id'],revision=ETTIN['revision'],attn_implementation='sdpa')
    assert backbone.config.model_type == 'modernbert'
    if CFG['gradient_checkpointing']:
        backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    return EmbeddingModel(backbone,tok,backbone.config.hidden_size,'ettin',backbone.config.to_dict())
if source_state is None:student=fresh_ettin().to(DEVICE)
else:
    spec=copy.deepcopy(source_state['spec']);typ=spec.pop('model_type');assert typ=='modernbert'
    backbone=AutoModel.from_config(AutoConfig.for_model(typ,**spec),attn_implementation='sdpa')
    tok=AutoTokenizer.from_pretrained(ETTIN['id'],revision=ETTIN['revision'])
    student=EmbeddingModel(backbone,tok,backbone.config.hidden_size,'ettin',source_state['spec']).to(DEVICE)
    student.load_state_dict(source_state['model'],strict=True);del source_state,backbone
    if CFG['gradient_checkpointing']:student.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
student.max_length=CFG['max_length'];student.requires_grad_(True)
assert DIMS==sorted(set(DIMS)) and DIMS[-1]==student.output_dim
assert CFG['max_length']<=student.backbone.config.max_position_embeddings
BASE_WEIGHTS={k:v.detach().cpu().clone() for k,v in student.state_dict().items()}
print(count_parameters(student))
# Auxiliary heads are training/evaluation only. They do not turn labels into retrieval pairs.
student.aux=nn.ModuleDict({task:nn.ModuleDict({str(d):nn.Linear(d,n) for d in DIMS})
                         for task,n in [('emotion',28),('topic',6)]}).to(DEVICE)
BASE_WEIGHTS={k:v.detach().cpu().clone() for k,v in student.state_dict().items()}


{'total': 32178048, 'backbone': 31883136, 'training_only_alignment': 294912}


In [31]:
class GTEEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(GTE['id'], revision=GTE['revision'])
        self.backbone = AutoModel.from_pretrained(GTE['id'], revision=GTE['revision'],
                                                  attn_implementation='sdpa', dtype=torch.bfloat16)
        self.output_dim = self.backbone.config.hidden_size
        self.max_length = CFG['max_length']
    def forward(self, texts):
        tokens = self.tokenizer(texts, padding=True, truncation=True, max_length=self.max_length,
                                return_tensors='pt').to(next(self.parameters()).device)
        # Published GTE-ModernBERT uses CLS pooling, not mean pooling.
        return self.backbone(**tokens).last_hidden_state[:,0].float()

supervisor = GTEEncoder().to(DEVICE).eval().requires_grad_(False)
print('Supervisor parameters:', sum(p.numel() for p in supervisor.parameters()))
@torch.no_grad()
def teacher_vectors(texts,length):
    supervisor.eval();supervisor.max_length=length
    parts=[]
    for start in range(0,len(texts),CFG['teacher_batch_size']):
        with torch.autocast('cuda',dtype=torch.bfloat16):z=supervisor(texts[start:start+CFG['teacher_batch_size']])
        parts.append(F.normalize(z.float(),dim=-1))
    result=torch.cat(parts)
    assert torch.isfinite(result).all()
    return result


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Supervisor parameters: 149014272


In [32]:
# A pinned, labeled retrieval benchmark. Never used for optimization or early stopping.
BENCHMARKS = {}
def load_benchmarks():
    if BENCHMARKS:
        return BENCHMARKS
    for name in BENCHMARK_NAMES:
        parts = {part: load_dataset('parquet', data_files={'train':f"hf://datasets/{NANO['id']}@{NANO['revision']}/{part}/{name}-*.parquet"}, split='train')
                 for part in ('queries', 'corpus', 'qrels')}
        queries = {str(r['_id']): r['text'] for r in parts['queries'] if r['text'].strip()}
        corpus = {str(r['_id']): r['text'] for r in parts['corpus'] if r['text'].strip()}
        relevant = defaultdict(set)
        for r in parts['qrels']:
            ids = r['corpus-id'] if isinstance(r['corpus-id'], list) else [r['corpus-id']]
            relevant[str(r['query-id'])].update(str(x) for x in ids)
        BENCHMARKS[name] = dict(queries=queries, corpus=corpus, relevant=dict(relevant))
    return BENCHMARKS

@torch.inference_mode()
def encode_texts(model, texts, batch_size=None, dim=None):
    model.eval()
    result = []
    for start in range(0, len(texts), batch_size or CFG['teacher_batch_size']):
        with torch.autocast('cuda', dtype=torch.bfloat16):
            z = model(texts[start:start+(batch_size or CFG['teacher_batch_size'])])
        z = z.float().cpu()
        assert torch.isfinite(z).all(), 'Non-finite embeddings'
        result.append(F.normalize(z[:, :dim], dim=-1) if dim else F.normalize(z, dim=-1))
    return torch.cat(result)

def retrieval_metrics(q, p, relevant, query_ids, corpus_ids):
    scores = q @ p.T
    ranking = scores.argsort(dim=1, descending=True)
    sums = defaultdict(float)
    for i, qid in enumerate(query_ids):
        truth = relevant[qid]
        top = [corpus_ids[j] for j in ranking[i, :10].tolist()]
        hits = [int(x in truth) for x in top]
        sums['hit@1'] += hits[0]
        sums['hit@5'] += bool(sum(hits[:5]))
        sums['recall@10'] += sum(hits) / len(truth)
        sums['mrr@10'] += next((1/(j+1) for j,h in enumerate(hits) if h), 0)
        dcg = sum(h/math.log2(j+2) for j,h in enumerate(hits))
        ideal = sum(1/math.log2(j+2) for j in range(min(len(truth), 10)))
        sums['ndcg@10'] += dcg / ideal
    return {k: v/len(query_ids) for k,v in sums.items()}

def benchmark(model, label, dims=None):
    results = {}
    for name, b in load_benchmarks().items():
        corpus_ids = list(b['corpus'])
        query_ids = [qid for qid in b['queries'] if b['relevant'].get(qid)]
        assert all(b['relevant'][qid] <= set(corpus_ids) for qid in query_ids)
        q = encode_texts(model, [b['queries'][k] for k in query_ids])
        p = encode_texts(model, list(b['corpus'].values()))
        for dim in dims or [q.shape[1]]:
            metrics = retrieval_metrics(F.normalize(q[:,:dim], dim=-1), F.normalize(p[:,:dim], dim=-1),
                                        b['relevant'], query_ids, corpus_ids)
            results[f'{name}/{dim}'] = metrics
            print(label, name, dim, {k: round(v,4) for k,v in metrics.items()}, flush=True)
    write_json(ROOT/'evaluations'/f'{label}.json', dict(label=label, contract=CONTRACT_ID,
               benchmark=NANO, max_length=model.max_length, results=results))
    return results


In [33]:
SANITY_CASES = [
    ('I forgot my account password.', 'Use the password reset link to regain access to your account.',
     'The tomato plants need watering every morning.'),
    ('How can I get my money back for this purchase?', 'Request a refund for your order through customer support.',
     'The train leaves the station at six in the evening.'),
    ('My phone battery runs out too quickly.', 'Reduce screen brightness and background apps to extend battery life.',
     'The museum has opened a new exhibition of ancient pottery.'),
    ('How do I bake a loaf of bread?', 'Mix flour, water and yeast, let the dough rise, then bake it in the oven.',
     'The football team won the championship yesterday.'),
    ('I want to cancel my monthly subscription.', 'Open your billing settings and choose cancel subscription.',
     'Migrating birds travel south during the winter.'),
    ('My laptop cannot connect to Wi-Fi.', 'Check the wireless network settings and restart your router.',
     'The orchestra performed a symphony at the concert hall.'),
    ('How can I track the package I ordered?', 'Enter your parcel tracking number to see its delivery status.',
     'Stretching your legs before a run can help improve flexibility.'),
    ('How do I change the language in the app?', 'Choose your preferred display language in the application settings.',
     'The recipe calls for two eggs and a cup of milk.'),
    ('I need to book a visit with a doctor.', 'Contact the clinic to schedule a medical appointment.',
     'The new telescope can observe distant galaxies.'),
    ('Where can I find books to borrow?', 'Visit the public library to check out books with your library card.',
     'The car needs new tires before the road trip.'),
    ('How can I recover a file that I deleted?', 'Look in the recycle bin or restore the document from a backup.',
     'The beach is covered in shells after the tide goes out.'),
    ('I want to learn to play the guitar.', 'Practice basic chords and strumming patterns on your guitar each day.',
     'The city council approved construction of a new bridge.'),
]

def sanity_scores(raw, cases, dimensions):
    # Pure scoring function: no model, tokenizer or teacher dependency.
    raw = raw.detach().float().cpu()
    n = len(cases)
    assert raw.ndim == 2 and len(raw) == 3*n
    finite = bool(torch.isfinite(raw).all())
    norms = raw.norm(dim=-1)
    nonzero = finite and bool((norms > 1e-8).all())
    result = dict(finite=finite, nonzero=nonzero,
                  raw_norm_min=float(norms.min()) if finite else None,
                  raw_norm_mean=float(norms.mean()) if finite else None,
                  profiles={})
    if not nonzero:
        return result
    for dim in dimensions:
        prefix_norms = raw[:, :dim].norm(dim=-1)
        z = F.normalize(raw[:, :dim], dim=-1)
        q, positive, unrelated = z.split(n)
        pos = (q*positive).sum(-1); neg = (q*unrelated).sum(-1)
        scores = q @ torch.cat((positive, unrelated)).T
        positive_mask = torch.zeros_like(scores, dtype=torch.bool)
        positive_mask[torch.arange(n), torch.arange(n)] = True
        # Conservative tie handling: a collapsed model must not pass by index order.
        ranks = 1 + ((scores >= pos[:,None]-1e-6) & ~positive_mask).sum(-1)
        pairwise = z @ z.T
        offdiag = pairwise[~torch.eye(len(z),dtype=torch.bool)]
        result['profiles'][str(dim)] = dict(
            nonzero_prefix=bool((prefix_norms > 1e-8).all()),
            easy_accuracy=float((pos > neg+1e-6).float().mean()),
            mean_margin=float((pos-neg).mean()),
            hit_at_1=float((ranks <= 1).float().mean()),
            hit_at_5=float((ranks <= 5).float().mean()),
            mrr=float((1/ranks.float()).mean()),
            normalized_spread=float((z-z.mean(0)).square().sum(-1).mean()),
            off_diagonal_cosine_mean=float(offdiag.mean()),
            examples=[dict(query=cases[i][0], positive=cases[i][1], unrelated=cases[i][2],
                           positive_cosine=float(pos[i]), unrelated_cosine=float(neg[i]),
                           positive_rank=int(ranks[i]), easy_correct=bool(pos[i]>neg[i]+1e-6))
                      for i in range(n)])
    return result


## Prepare all data before GPU training
Official GoEmotions splits; TweetTopic trains on 2020, validates on 2020, tests on 2021.
Retrieval holdouts are grouped deterministically. Twitter training pairs are grouped by URL and connected text
components, so a shared tweet or URL cannot leak between splits. Exact normalized text overlap is removed
across splits/sources, legacy training data, NanoBEIR, and the Twitter URL test set. This is not a guarantee
against semantic duplicates or unknown upstream pretraining contamination.

Emotion and topic examples retain their labels. They are never fabricated into same-label paraphrase pairs.
Missing downloads fail visibly. No source is silently substituted. Prepared data and identities are committed to Drive.


In [34]:
TEXT_KEYS=('query','positive','negative','text','sentence1','sentence2')
def row_ids(row):return {text_id(row[k]) for k in TEXT_KEYS if k in row}
def all_ids(rows):return {key for row in rows for key in row_ids(row)}
def hf_file(spec,filename):
    return hf_hub_download(spec['id'],filename,revision=spec['revision'],repo_type='dataset')

def read_twitter_test():
    path=hf_file(TWITTER_TEST,'test.json.gz')
    result=[]
    with gzip.open(path,'rt') as f:
        for line in f:
            raw=json.loads(line)
            for a,b,y in zip(raw['sent1'],raw['sent2'],raw['labels']):
                result.append(dict(sentence1=a,sentence2=b,label=int(y)))
    assert len(result)>100 and {r['label'] for r in result}=={0,1}
    return result
TWITTER_EVAL=read_twitter_test()
excluded={text_id(t) for b in load_benchmarks().values() for part in ('queries','corpus') for t in b[part].values()}
excluded.update(text_id(t) for case in SANITY_CASES for t in case)
excluded.update(all_ids(TWITTER_EVAL))
# Exclude legacy data from ALL newly defined splits, including source-run ancestors.
def legacy_rows(value):
    if isinstance(value,dict):
        if any(k in value for k in ('query','text','anchor')):yield value
        else:
            for v in value.values():yield from legacy_rows(v)
    elif isinstance(value,list):
        for v in value:yield from legacy_rows(v)
if INIT_MODE=='research_continuation':
    seen_runs=set();run=SOURCE_RUN
    while str(run) not in seen_runs:
        seen_runs.add(str(run))
        path=run/'data'/'mixture'/'pairs.json'
        if not path.exists():path=run/'pairs.json'
        if path.exists():excluded.update(all_ids(list(legacy_rows(json.loads(path.read_text())))))
        info=json.loads((run/'config.json').read_text()).get('source',{})
        if not info.get('run'):break
        run=Path(info['run'])

def clean(row,name):
    for k in TEXT_KEYS:
        if k in row:
            if not isinstance(row[k],str) or not row[k].strip() or len(row[k])>CFG['max_text_chars']:return None
            row[k]=row[k].strip()
    if row_ids(row)&excluded:return None
    row['_source']=name
    return row

def twitter_training():
    # The original TSV stores a vote tuple, not a binary label.
    rows=[]
    with open(TWITTER_TRAIN_FILE) as f:
        for line in f:
            if not line.strip():continue
            if line.lstrip().startswith('{'):
                r=json.loads(line);assert r['label'] in (0,1) and r.get('url')
            else:
                parts=line.rstrip('\n').split('\t');assert len(parts)>=4, 'Expected original Twitter URL TSV with URL'
                votes=re.fullmatch(r'\s*\(\s*(\d+)\s*,\s*6\s*\)\s*',parts[2])
                assert votes, f'Invalid vote tuple: {parts[2]}'
                n=int(votes[1]);assert 0<=n<=6
                if n==3:continue
                r=dict(sentence1=parts[0],sentence2=parts[1],label=int(n>=4),url=parts[3])
            r=clean(r,'twitter_url')
            if r:rows.append(r)
    # Union URL groups and repeated text identities before assigning a split.
    parent={}
    def find(x):
        parent.setdefault(x,x)
        while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]
        return x
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b:parent[max(a,b)]=min(a,b)
    for r in rows:
        ids=sorted(row_ids(r));a=ids[0];u='url:'+text_id(r['url']);union(a,ids[-1]);union(a,u)
    result={s:[] for s in ('train','validation','test')}
    for r in rows:
        group=find(next(iter(row_ids(r))));bucket=int(digest(group)[:8],16)%100
        split='test' if bucket<10 else 'validation' if bucket<20 else 'train'
        result[split].append(r)
    return result

def collect_source(name):
    if name=='twitter_url':return twitter_training()
    spec=DATA_SPECS[name];result={s:[] for s in ('train','validation','test')}
    if name in ('go_emotions','tweet_topic'):
        for split in result:
            if name=='go_emotions':
                ds=load_dataset(spec['id'],'simplified',revision=spec['revision'],split=split)
            else:
                filename={'train':'train_2020','validation':'validation_2020','test':'test_2021'}[split]
                path=hf_file(spec,f'dataset/split_temporal/{filename}.single.json')
                ds=[json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
            order=np.random.default_rng(CFG['seed']).permutation(len(ds))
            cap=SOURCE_CAPS[name] if split=='train' else CFG[split+'_rows']
            for i in order:
                raw=ds[int(i)];r=clean(dict(text=raw['text'],labels=raw['labels']) if name=='go_emotions'
                                     else dict(text=raw['text'],label=int(raw['label'])),name)
                if r:result[split].append(r)
                if len(result[split])>=2*cap:break
        return result
    caps=dict(train=2*SOURCE_CAPS[name],validation=2*CFG['validation_rows'],test=2*CFG['test_rows'])
    stream=load_dataset(spec['id'],spec['subset'],revision=spec['revision'],split='train',streaming=True)
    stream=stream.shuffle(seed=CFG['seed'],buffer_size=CFG['shuffle_buffer']);seen=set()
    for i,raw in enumerate(stream,1):
        if name=='nq':r=dict(query=raw['query'],positive=raw['answer'])
        elif name=='squad':r=dict(query=raw['question'],positive=raw['answer'])
        elif name=='msmarco':r={k:raw[k] for k in ('query','positive','negative')}
        else:r=dict(query=raw['anchor'],positive=raw['positive'],negative=raw['negative'])
        r=clean(r,name)
        if r and len(row_ids(r))==len([k for k in TEXT_KEYS if k in r]):
            group=text_id(r['query'] if name in ('all_nli','msmarco') else r['positive'])
            bucket=int(group[:8],16)%100;split='test' if bucket<5 else 'validation' if bucket<10 else 'train'
            q=text_id(r['query'])
            if q not in seen and len(result[split])<caps[split]:
                seen.add(q)
                if 'negative' in r:r['_negative_kind']='labeled'
                result[split].append(r)
        if all(len(result[s])>=caps[s] for s in caps) or i>=CFG['scan_limit']:break
    return result


In [35]:
def mine_lexical_negatives(rows,name):
    # Train-only candidate pool; labels are not inferred from mere lexical similarity.
    docs=list(dict.fromkeys(r['positive'] for r in rows));rng=np.random.default_rng(CFG['seed'])
    docs=[docs[i] for i in rng.permutation(len(docs))[:CFG['mining_pool']]]
    vectorizer=TfidfVectorizer(max_features=50000,ngram_range=(1,2),sublinear_tf=True)
    matrix=vectorizer.fit_transform(docs);docids=[text_id(t) for t in docs]
    known=defaultdict(set)
    for row in rows:known[text_id(row['query'])].add(text_id(row['positive']))
    for start in range(0,len(rows),128):
        block=rows[start:start+128];scores=(vectorizer.transform([r['query'] for r in block])@matrix.T).toarray()
        for row,s in zip(block,scores):
            forbidden=known[text_id(row['query'])]|{text_id(row['query'])}
            for j,key in enumerate(docids):
                if key in forbidden:s[j]=-np.inf
            valid=np.flatnonzero(np.isfinite(s));assert len(valid), 'Need multiple distinct documents to mine negatives.'
            k=min(CFG['mining_candidates'],len(valid));top=valid[np.argpartition(s[valid],-k)[-k:]]
            pick=int(top[rng.integers(len(top))]);row['negative']=docs[pick];row['_negative_kind']='lexical_candidate'
        if start%10000==0:print(name,'mined',min(start+128,len(rows)),'/',len(rows),flush=True)
    return rows


In [36]:
DATA_DIR=ROOT/'data'/'mixture'
def audit_data(data):
    sets={s:set() for s in ('train','validation','test')}
    for name,parts in data.items():
        for split,rows in parts.items():
            assert len(rows)>= (max(p['batch'] for p in PHASES.values()) if split=='train' else 2), (name,split,len(rows))
            ids=all_ids(rows);assert not ids&excluded;sets[split].update(ids)
            if TASKS[name]=='paraphrase':assert {r['label'] for r in rows}=={0,1}
    assert not (sets['train']&sets['validation'] or sets['train']&sets['test'] or sets['validation']&sets['test'])
if DATA_DIR.exists():
    meta=json.loads((DATA_DIR/'COMMITTED.json').read_text())
    assert meta['contract']==CONTRACT_ID and file_sha(DATA_DIR/'pairs.json')==meta['sha256']
    corpus=json.loads((DATA_DIR/'pairs.json').read_text())
else:
    candidates={n:collect_source(n) for n in SOURCE_CAPS}
    corpus={n:{s:[] for s in ('train','validation','test')} for n in SOURCE_CAPS}
    reserved=set(excluded)
    for split in ('test','validation','train'):
        for name in SOURCE_CAPS:
            accepted=set();seen=set();cap=SOURCE_CAPS[name] if split=='train' else CFG[split+'_rows']
            for row in candidates[name][split]:
                ids=row_ids(row);identity=digest(sorted(ids))
                if ids&reserved or identity in seen:continue
                seen.add(identity);corpus[name][split].append(row);accepted.update(ids)
                if len(corpus[name][split])>=cap:break
            reserved.update(accepted)
    del candidates
    for name in ('nq','squad'):mine_lexical_negatives(corpus[name]['train'],name)
    audit_data(corpus)
    tmp=DATA_DIR.parent/('.uploading-'+uuid.uuid4().hex);tmp.mkdir(parents=True)
    write_json(tmp/'pairs.json',corpus)
    write_json(tmp/'COMMITTED.json',dict(contract=CONTRACT_ID,sha256=file_sha(tmp/'pairs.json')))
    os.replace(tmp,DATA_DIR)
audit_data(corpus);DATA_ID=digest(corpus)
write_json(ROOT/'data-summary.json',dict(data_id=DATA_ID,tasks=TASKS,
    twitter_training=bool(TWITTER_TRAIN_FILE),counts={n:{s:len(r) for s,r in p.items()} for n,p in corpus.items()}))
print('Actual counts:',{n:{s:len(r) for s,r in p.items()} for n,p in corpus.items()})
# Audit actual lengths; do not repeatedly oversample a handful of long examples.
length_report={}
for name,parts in corpus.items():
    texts=[r.get('positive',r.get('text',r.get('sentence1'))) for r in parts['train']]
    lengths=[]
    for start in range(0,len(texts),512):
        lengths.extend(student.tokenizer(texts[start:start+512],truncation=False,return_length=True)['length'])
    length_report[name]=dict(n=len(lengths),p50=float(np.percentile(lengths,50)),p95=float(np.percentile(lengths,95)),
                            truncated_fraction=float(np.mean(np.array(lengths)>CFG['short_length'])))
write_json(ROOT/'length-report.json',length_report);print(length_report)


nq mined 128 / 20000
squad mined 128 / 20000
Actual counts: {'nq': {'train': 20000, 'validation': 128, 'test': 256}, 'squad': {'train': 20000, 'validation': 128, 'test': 256}, 'all_nli': {'train': 30000, 'validation': 128, 'test': 256}, 'msmarco': {'train': 60000, 'validation': 128, 'test': 256}, 'go_emotions': {'train': 30000, 'validation': 128, 'test': 256}, 'tweet_topic': {'train': 2858, 'validation': 128, 'test': 256}}
{'nq': {'n': 20000, 'p50': 126.0, 'p95': 275.0, 'truncated_fraction': 0.0701}, 'squad': {'n': 20000, 'p50': 129.0, 'p95': 276.0, 'truncated_fraction': 0.07}, 'all_nli': {'n': 30000, 'p50': 9.0, 'p95': 17.0, 'truncated_fraction': 0.0}, 'msmarco': {'n': 60000, 'p50': 73.0, 'p95': 145.0, 'truncated_fraction': 0.0002666666666666667}, 'go_emotions': {'n': 30000, 'p50': 18.0, 'p95': 33.0, 'truncated_fraction': 0.0}, 'tweet_topic': {'n': 2858, 'p50': 45.0, 'p95': 81.0, 'truncated_fraction': 0.0}}


## Losses and validation
Stage 1 trains the full vector; stage 2 trains all dimension prefixes. Both update the backbone.
Retrieval uses teacher-filtered InfoNCE plus GTE alignment/geometry. Social classification uses separate
training-only heads on normalized prefixes; Twitter paraphrases use labeled cosine margin loss.
Classification batches also receive GTE geometry/alignment to reduce drift. No sentiment-based pseudo-paraphrases.
Validation reports task-appropriate scores (retrieval MRR, emotion micro-F1, topic macro-F1, paraphrase AP).
Selection averages sources equally and protects mean full-width retrieval MRR within the configured margin.
These mixed scores are a selection heuristic, not a universal accuracy. Tests never select checkpoints.


In [37]:
def flatten_pairs(rows):
    assert all('negative' in r for r in rows)
    return [r[k] for k in ('query','positive','negative') for r in rows]

def negative_mask(rows,target):
    n=len(rows);docids=[text_id(r[k]) for k in ('positive','negative') for r in rows]
    qids=[text_id(r['query']) for r in rows];posids=[text_id(r['positive']) for r in rows]
    mask=torch.ones((n,2*n),dtype=torch.bool,device=target.device)
    for i in range(n):
        for j in range(2*n):
            if j!=i and (docids[j]==posids[i] or (j%n!=i and qids[j%n]==qids[i])):mask[i,j]=False
    scores=target[:n]@target[n:].T
    positive=scores[torch.arange(n,device=target.device),torch.arange(n,device=target.device)]
    suspected=scores>=positive[:,None]-CFG['false_negative_margin']
    for i,r in enumerate(rows):
        suspected[i,i]=False
        if r.get('_negative_kind')=='labeled':suspected[i,n+i]=False
    mask &= ~suspected
    mask[torch.arange(n),torch.arange(n)]=True
    return mask

@torch.autocast('cuda',enabled=False)
def retrieval_objective(model,raw,target,rows):
    n=len(rows);target=F.normalize(target.float(),dim=-1);mask=negative_mask(rows,target)
    geometry=target@target.T;offdiag=~torch.eye(len(raw),device=raw.device,dtype=torch.bool)
    rank_terms=[];relations=[]
    for dim in ACTIVE_DIMS:
        z=F.normalize(raw[:,:dim].float(),dim=-1)
        logits=(z[:n]@z[n:].T/CFG['temperature']).masked_fill(~mask,torch.finfo(torch.float32).min)
        rank_terms.append(F.cross_entropy(logits,torch.arange(n,device=raw.device)))
        relations.append(F.mse_loss((z@z.T)[offdiag],geometry[offdiag]))
    ranking=torch.stack(rank_terms).mean();relation=torch.stack(relations).mean()
    aligned=F.normalize(model.align(raw).float(),dim=-1)
    align=(1-(aligned*target).sum(-1)).mean()
    loss=ranking+CFG['alignment_weight']*align+CFG['relational_weight']*relation
    return loss,dict(ranking=float(ranking.detach()),alignment=float(align.detach()),relation=float(relation.detach()),
                     negatives_per_query=float((mask.sum(1)-1).float().mean()))


In [38]:
ACTIVE_DIMS=DIMS

def batch_texts(name,rows):
    task=TASKS[name]
    if task=='retrieval':return flatten_pairs(rows)
    if task=='paraphrase':return [r[k] for k in ('sentence1','sentence2') for r in rows]
    return [r['text'] for r in rows]

@torch.autocast('cuda',enabled=False)
def social_loss(raw,name,rows,dims):
    task=TASKS[name];terms=[]
    for dim in dims:
        z=F.normalize(raw[:,:dim].float(),dim=-1)
        if task=='paraphrase':
            a,b=z.chunk(2);cos=(a*b).sum(-1);y=torch.tensor([r['label'] for r in rows],device=z.device)
            # Non-paraphrases may still be related: a margin, not forced cosine -1.
            terms.append(torch.where(y.bool(),1-cos,F.relu(cos-.3)).mean())
        else:
            logits=student.aux[task][str(dim)](z)
            if task=='emotion':
                y=torch.zeros_like(logits)
                for i,r in enumerate(rows):y[i,r['labels']]=1
                terms.append(F.binary_cross_entropy_with_logits(logits,y))
            else:terms.append(F.cross_entropy(logits,torch.tensor([r['label'] for r in rows],device=z.device)))
    return torch.stack(terms).mean()

@torch.autocast('cuda',enabled=False)
def backbone_loss(model,name,rows):
    texts=batch_texts(name,rows);target=teacher_vectors(texts,CFG['short_length'])
    with torch.autocast('cuda',dtype=torch.bfloat16):raw=model(texts)
    if TASKS[name]=='retrieval':return retrieval_objective(model,raw.float(),target,rows)
    supervised=social_loss(raw,name,rows,ACTIVE_DIMS)
    z=F.normalize(raw.float(),dim=-1);t=target.float()
    aligned=F.normalize(model.align(raw.float()),dim=-1)
    alignment=(1-(aligned*t).sum(-1)).mean()
    relation=F.mse_loss(z@z.T,t@t.T)
    loss=CFG['aux_weight']*supervised+CFG['alignment_weight']*alignment+CFG['relational_weight']*relation
    return loss,dict(supervised=float(supervised.detach()),alignment=float(alignment.detach()))

@torch.inference_mode()
def task_metrics(vectors,name,rows,dims):
    task=TASKS[name];profiles=[];n=len(rows)
    for dim in dims:
        z=F.normalize(vectors[:,:dim].float(),dim=-1)
        if task=='retrieval':
            q,p=z[:n],z[n:];ids=[text_id(r['positive']) for r in rows]
            relevant=torch.tensor([[a==b for b in ids] for a in ids]);sim=q@p.T
            positive=sim.masked_fill(~relevant,-float('inf')).max(1).values
            rank=1+((sim>=positive[:,None]-1e-6)&~relevant).sum(1)
            metric=dict(mrr=float((1/rank.float()).mean()),hit1=float((rank==1).float().mean()));score=metric['mrr']
        elif task=='paraphrase':
            a,b=z.chunk(2);scores=(a*b).sum(-1).numpy();y=[r['label'] for r in rows]
            metric=dict(average_precision=float(average_precision_score(y,scores)),auc=float(roc_auc_score(y,scores)))
            score=metric['average_precision']
        else:
            head=student.aux[task][str(dim)]
            logits=F.linear(z,head.weight.detach().float().cpu(),head.bias.detach().float().cpu()).numpy()
            if task=='emotion':
                y=np.zeros_like(logits,dtype=int)
                for i,r in enumerate(rows):y[i,r['labels']]=1
                pred=logits>0;metric=dict(micro_f1=float(f1_score(y,pred,average='micro',zero_division=0)),
                                        macro_f1=float(f1_score(y,pred,average='macro',zero_division=0)));score=metric['micro_f1']
            else:
                y=[r['label'] for r in rows];pred=logits.argmax(1)
                metric=dict(macro_f1=float(f1_score(y,pred,labels=list(range(6)),average='macro',zero_division=0)),accuracy=float(accuracy_score(y,pred)))
                score=metric['macro_f1']
        profiles.append(dict(dim=dim,score=score,**metric))
    return profiles

def eval_texts(name,rows):
    if TASKS[name]=='retrieval':return [r[k] for k in ('query','positive') for r in rows]
    return batch_texts(name,rows)

@torch.inference_mode()
def validate_model(model,split='validation'):
    result={};full=[]
    for name,parts in corpus.items():
        rows=parts[split];vectors=encode_texts(model,eval_texts(name,rows),batch_size=CFG['eval_batch_size'])
        profiles=task_metrics(vectors,name,rows,ACTIVE_DIMS)
        result[name]=dict(n=len(rows),profiles=profiles,score=float(np.mean([p['score'] for p in profiles])))
        if TASKS[name]=='retrieval':full += [p['mrr'] for p in profiles if p['dim']==DIMS[-1]]
    return dict(selection_score=float(np.mean([x['score'] for x in result.values()])),full_score=float(np.mean(full)),by_source=result)


In [39]:
# Completed generations are immutable. Upload to a temporary directory; COMMITTED.json is written last.
# Resume discovers the newest verified generation, rather than trusting a possibly stale "latest" pointer.
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), cpu=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all())
def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy'])
    torch.set_rng_state(state['cpu'])
    if torch.cuda.is_available(): torch.cuda.set_rng_state_all(state['cuda'])

def save_checkpoint(stage, step, model, optimizer, scheduler, sampler, best_metric, best_step, total_steps):
    parent=ROOT/'checkpoints'/stage; parent.mkdir(parents=True,exist_ok=True)
    generation=f'{step:08d}-{uuid.uuid4().hex[:8]}'
    local=LOCAL/(generation+'.pt')
    torch.save(dict(stage=stage,step=step,total_steps=total_steps,contract=CONTRACT_ID,data_id=DATA_ID,
                    family=model.family,spec=model.spec,model=model.state_dict(),optimizer=optimizer.state_dict(),
                    scheduler=scheduler.state_dict(),sampler=sampler.state_dict(),rng=rng_state(),
                    best_metric=best_metric,best_step=best_step),local)
    tmp=parent/('.uploading-'+generation); tmp.mkdir()
    shutil.copyfile(local,tmp/'state.pt')
    sha=file_sha(local)
    assert file_sha(tmp/'state.pt') == sha, 'Drive checkpoint copy failed checksum verification'
    write_json(tmp/'COMMITTED.json',dict(step=step,sha256=sha,contract=CONTRACT_ID))
    os.replace(tmp,parent/generation)
    local.unlink()
    # Retain latest two plus the generation containing the best validation result.
    generations=sorted(p for p in parent.iterdir() if (p/'COMMITTED.json').exists() and not p.name.startswith('.'))
    keep={p.name for p in generations[-2:]}
    keep.update(p.name for p in generations if int(p.name.split('-')[0]) == best_step)
    for p in generations:
        if p.name not in keep: shutil.rmtree(p)
    return parent/generation

def latest_checkpoint(stage):
    parent=ROOT/'checkpoints'/stage
    if not parent.exists(): return None
    candidates=sorted((p for p in parent.iterdir() if not p.name.startswith('.') and (p/'COMMITTED.json').exists()),reverse=True)
    for path in candidates:
        try:
            manifest=json.loads((path/'COMMITTED.json').read_text())
            assert manifest['contract']==CONTRACT_ID
            assert file_sha(path/'state.pt')==manifest['sha256']
            return path
        except (OSError,ValueError,AssertionError) as e:
            print('Skipping invalid checkpoint',path,e)
    if candidates: raise RuntimeError('No valid checkpoint remains; inspect Drive files.')
    return None

def load_checkpoint(stage):
    path=latest_checkpoint(stage)
    if path is None: return None
    local=LOCAL/(stage+'-resume.pt'); shutil.copyfile(path/'state.pt',local)
    # Only our own checkpoints, never arbitrary downloaded pickle files.
    assert file_sha(local)==json.loads((path/'COMMITTED.json').read_text())['sha256']
    state=torch.load(local,map_location='cpu',weights_only=False)
    assert state['contract']==CONTRACT_ID and state['data_id']==DATA_ID
    return state


In [40]:
class RowSampler:
    def __init__(self,n,seed):
        assert n>0
        self.n=n;self.generator=torch.Generator().manual_seed(seed)
        self.order=torch.randperm(n,generator=self.generator);self.cursor=0;self.epoch=0;self.drawn=0
    def next(self,batch):
        assert batch<=self.n
        ids=[]
        while len(ids)<batch:
            if self.cursor==self.n:
                self.order=torch.randperm(self.n,generator=self.generator);self.cursor=0;self.epoch+=1
            size=min(batch-len(ids),self.n-self.cursor)
            ids.extend(self.order[self.cursor:self.cursor+size].tolist());self.cursor+=size
        self.drawn+=batch;return ids
    def state_dict(self):
        return dict(n=self.n,order=self.order,cursor=self.cursor,epoch=self.epoch,drawn=self.drawn,generator=self.generator.get_state())
    def load_state_dict(self,state):
        assert state['n']==self.n and 0<=state['cursor']<=self.n
        self.order=state['order'].clone();self.cursor=state['cursor'];self.epoch=state['epoch'];self.drawn=state['drawn'];self.generator.set_state(state['generator'])


In [41]:
class TaskSampler:
    def __init__(self,batch):
        self.batch=batch;self.turn=0
        self.samplers={n:RowSampler(len(corpus[n]['train']),CFG['seed']+i) for i,n in enumerate(corpus)}
    def next(self):
        name=MIX_CYCLE[self.turn%len(MIX_CYCLE)];self.turn+=1
        return name,[corpus[name]['train'][i] for i in self.samplers[name].next(self.batch)]
    def state_dict(self):return dict(batch=self.batch,turn=self.turn,samplers={n:s.state_dict() for n,s in self.samplers.items()})
    def load_state_dict(self,state):
        assert state['batch']==self.batch and set(state['samplers'])==set(self.samplers)
        self.turn=state['turn']
        for n,s in self.samplers.items():s.load_state_dict(state['samplers'][n])


In [42]:
def restore_best(model,stage):
    state=load_checkpoint(stage)
    assert state is not None
    best=state['best_step']; del state
    parent=ROOT/'checkpoints'/stage
    choices=sorted(p for p in parent.iterdir() if p.name.startswith(f'{best:08d}-') and (p/'COMMITTED.json').exists())
    assert choices, 'Best checkpoint is missing'
    path=choices[-1];meta=json.loads((path/'COMMITTED.json').read_text())
    assert file_sha(path/'state.pt')==meta['sha256']
    state=torch.load(path/'state.pt',map_location='cpu',weights_only=False)
    model.load_state_dict(state['model'],strict=True)
    print('Selected',stage,'validation-best step',best)
    return model


In [43]:
def train_stage(model,stage,loss_fn,validation_fn):
    global ACTIVE_DIMS
    phase=PHASES[stage];ACTIVE_DIMS=phase['dims'];steps=phase['steps']
    model.to(DEVICE)
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=phase['lr'],weight_decay=.01)
    def multiplier(step):
        warm=min(100,max(1,steps//10))
        if step<warm:return (step+1)/warm
        ratio=min(1.,(step-warm)/max(1,steps-warm))
        return .1+.9*.5*(1+math.cos(math.pi*ratio))
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,multiplier)
    sampler=TaskSampler(phase['batch']);state=load_checkpoint(stage)
    baseline_path=ROOT/'evaluations'/f'{stage}-baseline.json'
    if baseline_path.exists():
        baseline_record=json.loads(baseline_path.read_text());assert baseline_record['contract']==CONTRACT_ID
        baseline=baseline_record['metrics']
    else:
        baseline=validation_fn(model)
        write_json(baseline_path,dict(contract=CONTRACT_ID,metrics=baseline))
    start=0;best=baseline['selection_score'];best_step=0
    if state:
        assert state['spec']==model.spec and state['total_steps']==steps
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler'])
        sampler.load_state_dict(state['sampler']);restore_rng(state['rng'])
        start=state['step'];best=state['best_metric'];best_step=state['best_step'];del state
        print(stage,'resuming',start,'/',steps)
    else:save_checkpoint(stage,0,model,optimizer,scheduler,sampler,best,0,steps)
    last=start;mutating=False
    try:
        for step in range(start+1,steps+1):
            boundary=copy.deepcopy(sampler.state_dict());boundary_rng=rng_state()
            name,rows=sampler.next();model.train();optimizer.zero_grad(set_to_none=True)
            loss,parts=loss_fn(model,name,rows)
            assert torch.isfinite(loss),'Non-finite loss'
            if stage!='exits':loss.backward()  # exits accumulate one branch backward at a time
            nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True)
            mutating=True;optimizer.step();scheduler.step();last=step;mutating=False
            record=dict(stage=stage,step=step,source=name,loss=float(loss.detach()),**parts);improved=False
            if step==1 or step%CFG['log_every']==0:print(stage,step,'/',steps,name,'loss=',round(record['loss'],5),flush=True)
            if step%CFG['eval_every']==0 or step==steps:
                metrics=validation_fn(model);record['validation']=metrics
                eligible=metrics['full_score']>=baseline['full_score']-CFG['max_full_mrr_drop']
                if eligible and metrics['selection_score']>best:
                    best=metrics['selection_score'];best_step=step;improved=True
                print(stage,'validation',round(metrics['selection_score'],4),'best',best_step,flush=True)
            with open(ROOT/f'{stage}-metrics.jsonl','a') as f:f.write(json.dumps(record)+'\n')
            if improved or step%CFG['checkpoint_every']==0 or step==steps:
                save_checkpoint(stage,step,model,optimizer,scheduler,sampler,best,best_step,steps)
    except KeyboardInterrupt:
        if not mutating:
            if last<step:sampler.load_state_dict(boundary);restore_rng(boundary_rng)
            optimizer.zero_grad(set_to_none=True)
            save_checkpoint(stage,last,model,optimizer,scheduler,sampler,best,best_step,steps)
        print('Interrupted. Rerun this stage to restore its last committed update.');raise
    model=restore_best(model,stage).eval()
    selection=dict(step=best_step,score=best,completed_steps=last)
    write_json(ROOT/f'{stage}-selected.json',selection)
    write_json(ROOT/'evaluations'/f'{stage}-test.json',validation_fn(model,'test'))
    return model,selection


## Stage 1 — adapt the full-width backbone


In [ ]:
student.max_length=CFG['short_length'];student.requires_grad_(True)
student,BACKBONE_SELECTED=train_stage(student,'backbone',backbone_loss,validate_model)


backbone 1 / 10000 msmarco loss= 0.80985
backbone 25 / 10000 squad loss= 1.03012
backbone 50 / 10000 msmarco loss= 0.64177
backbone 75 / 10000 nq loss= 1.10215
backbone 100 / 10000 msmarco loss= 0.59151
backbone 125 / 10000 go_emotions loss= 0.33044
backbone 150 / 10000 all_nli loss= 1.6669
backbone 175 / 10000 tweet_topic loss= 0.63551
backbone 200 / 10000 go_emotions loss= 0.31886
backbone 225 / 10000 tweet_topic loss= 0.62459
backbone 250 / 10000 squad loss= 0.99883
backbone validation 0.6061 best 250
backbone 275 / 10000 msmarco loss= 0.60013
backbone 300 / 10000 nq loss= 0.58369
backbone 325 / 10000 msmarco loss= 0.46553
backbone 350 / 10000 go_emotions loss= 0.31317
backbone 375 / 10000 all_nli loss= 1.00012
backbone 400 / 10000 tweet_topic loss= 0.60519
backbone 425 / 10000 go_emotions loss= 0.30821
backbone 450 / 10000 tweet_topic loss= 0.60842
backbone 475 / 10000 squad loss= 0.97447
backbone 500 / 10000 msmarco loss= 0.40429
backbone validation 0.6117 best 500
backbone 525 / 

## Stage 2 — Matryoshka dimension prefixes, with full-width retrieval protection


In [ ]:
student,PREFIX_SELECTED=train_stage(student,'prefix',backbone_loss,validate_model)
# Release GTE before training exit branches. Full-depth student is now the fixed teacher.
supervisor.cpu();del supervisor;gc.collect();torch.cuda.empty_cache()
student.eval().requires_grad_(False);student.backbone.gradient_checkpointing_disable()
backbone=student.backbone;tokenizer=student.tokenizer
WIDTH=student.output_dim;NUM_LAYERS=backbone.config.num_hidden_layers
DEPTHS=list(range(1,NUM_LAYERS)) if EXIT_DEPTHS is None else list(EXIT_DEPTHS)
assert DEPTHS==sorted(set(DEPTHS)) and all(1<=d<NUM_LAYERS for d in DEPTHS)
def backbone_digest():
    h=hashlib.sha256()
    for k,v in backbone.state_dict().items():
        h.update(k.encode());h.update(v.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())
    return h.hexdigest()
BACKBONE_SHA=backbone_digest()
SOURCE_MODEL_SPEC=dict(kind='transformer',depths=DEPTHS,width=WIDTH,backbone_sha=BACKBONE_SHA,
                       prefix_selected=PREFIX_SELECTED,independent_heads=True)


## Exact early-exit transformer execution
Every exit starts as a copy of the selected backbone's final transformer block. All heads are independent.
The backbone is frozen. Training obtains all intermediate states in one full forward pass; inference at depth L
runs only L backbone blocks plus that exit's block. Shared-index validation uses full-depth document embeddings.


In [ ]:
def tokenize(texts):
    return tokenizer(texts,padding=True,truncation=True,max_length=CFG['short_length'],return_tensors='pt').to(DEVICE)

@torch.no_grad()
def frozen_features(tokens,depths):
    backbone.eval()
    last=max(depths);assert 1<=last<=NUM_LAYERS
    h=backbone.embeddings(input_ids=tokens['input_ids'])
    positions=torch.arange(h.shape[1],device=h.device).unsqueeze(0)
    mask_args=dict(config=backbone.config,input_embeds=h,attention_mask=tokens['attention_mask'])
    masks={'full_attention':create_bidirectional_mask(**mask_args),
           'sliding_attention':create_bidirectional_sliding_window_mask(**mask_args)}
    rope={kind:backbone.rotary_emb(h,positions,kind) for kind in set(backbone.config.layer_types)}
    outputs={}
    for depth,layer in enumerate(backbone.layers[:last],1):
        h=layer(h,attention_mask=masks[layer.attention_type],position_embeddings=rope[layer.attention_type])
        if depth in depths:outputs[depth]=h.detach()
    return outputs,dict(masks=masks,rope=rope)

def native_pool(hidden,mask):
    return mean_pool(backbone.final_norm(hidden),mask)


class ExitHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm=copy.deepcopy(backbone.final_norm).requires_grad_(False)
        self.block=copy.deepcopy(backbone.layers[-1]).requires_grad_(True)
        self.block.gradient_checkpointing=False
    def forward(self,h,mask,context):
        typ=self.block.attention_type
        h=self.block(h,attention_mask=context['masks'][typ],position_embeddings=context['rope'][typ])
        return mean_pool(self.norm(h),mask)

class ExitBank(nn.Module):
    def __init__(self):
        super().__init__();self.family='ettin-exits';self.spec=copy.deepcopy(SOURCE_MODEL_SPEC)
        self.heads=nn.ModuleDict({str(d):ExitHead() for d in DEPTHS})
    def forward(self,depth,h,mask,context):return self.heads[str(depth)](h,mask,context)

def early_raw(tokens,depth,bank=None):
    features,context=frozen_features(tokens,[depth])
    if depth==NUM_LAYERS or bank is None:return native_pool(features[depth],tokens['attention_mask'])
    return bank(depth,features[depth],tokens['attention_mask'],context)

@torch.inference_mode()
def encode_profiles(texts,bank=None,batch_size=None):
    if bank is not None:bank.eval()
    result={d:[] for d in DEPTHS+[NUM_LAYERS]}
    for start in range(0,len(texts),batch_size or CFG['teacher_batch_size']):
        tokens=tokenize(texts[start:start+(batch_size or CFG['teacher_batch_size'])])
        with torch.autocast('cuda',dtype=torch.bfloat16):
            features,context=frozen_features(tokens,DEPTHS+[NUM_LAYERS])
            for depth in result:
                z=native_pool(features[depth],tokens['attention_mask']) if bank is None or depth==NUM_LAYERS else bank(depth,features[depth],tokens['attention_mask'],context)
                z=z.float().cpu()
                assert torch.isfinite(z).all() and (z.norm(dim=-1)>1e-8).all(), 'Invalid embeddings'
                result[depth].append(z)
    return {depth:torch.cat(parts) for depth,parts in result.items()}


In [ ]:
probe=['Recover a forgotten password.','Books can be borrowed at the library.']
tokens=tokenize(probe)
with torch.inference_mode(),torch.autocast('cuda',dtype=torch.bfloat16):
    canonical=mean_pool(backbone(**tokens).last_hidden_state,tokens['attention_mask'])
    torch.testing.assert_close(canonical,early_raw(tokens,NUM_LAYERS),atol=1e-5,rtol=1e-4)
counts=[0]*NUM_LAYERS
hooks=[layer.register_forward_hook(lambda module,args,output,i=i:counts.__setitem__(i,counts[i]+1)) for i,layer in enumerate(backbone.layers)]
try:
    with torch.inference_mode(),torch.autocast('cuda',dtype=torch.bfloat16):early_raw(tokens,min(DEPTHS))
finally:
    for hook in hooks:hook.remove()
assert counts==[int(i<min(DEPTHS)) for i in range(NUM_LAYERS)]
print('Canonical full-depth parity and actual early-exit execution passed.')


In [ ]:
@torch.autocast('cuda',enabled=False)
def exit_loss(bank,name,rows):
    tokens=tokenize(batch_texts(name,rows))
    with torch.no_grad(),torch.autocast('cuda',dtype=torch.bfloat16):
        features,context=frozen_features(tokens,DEPTHS+[NUM_LAYERS])
        target=native_pool(features[NUM_LAYERS],tokens['attention_mask']).float()
    # One head at a time, accumulating gradients, avoids retaining nine transformer graphs.
    # The stage driver clips/steps after these backwards; it must not call backward again.
    total=0.
    for depth in DEPTHS:
        with torch.autocast('cuda',dtype=torch.bfloat16):raw=bank(depth,features[depth],tokens['attention_mask'],context)
        terms=[]
        for dim in DIMS:
            z=F.normalize(raw[:,:dim].float(),dim=-1);t=F.normalize(target[:,:dim],dim=-1)
            terms.append((1-(z*t).sum(-1)).mean()+.2*F.mse_loss(z@z.T,t@t.T))
        alignment=torch.stack(terms).mean()
        if TASKS[name]=='retrieval':
            n=len(rows);mask=negative_mask(rows,F.normalize(target,dim=-1));ranking=[]
            for dim in DIMS:
                z=F.normalize(raw[:,:dim].float(),dim=-1)
                logits=(z[:n]@z[n:].T/CFG['temperature']).masked_fill(~mask,-1e9)
                ranking.append(F.cross_entropy(logits,torch.arange(n,device=raw.device)))
            supervised=torch.stack(ranking).mean()
        else:supervised=CFG['aux_weight']*social_loss(raw,name,rows,DIMS)
        loss=(alignment+supervised)/len(DEPTHS)
        assert torch.isfinite(loss)
        loss.backward();total+=float(loss.detach())
    return torch.tensor(total,device=DEVICE),dict(heads=len(DEPTHS))

@torch.inference_mode()
def validate_exits(bank,split='validation'):
    results={};full=[]
    for name,parts in corpus.items():
        rows=parts[split];n=len(rows);vectors=encode_profiles(eval_texts(name,rows),bank,batch_size=CFG['eval_batch_size'])
        profiles=[]
        for depth in DEPTHS+[NUM_LAYERS]:
            for p in task_metrics(vectors[depth],name,rows,DIMS):profiles.append(dict(depth=depth,mode='same_exit',**p))
            if TASKS[name]=='retrieval' and depth!=NUM_LAYERS:
                shared=torch.cat([vectors[depth][:n],vectors[NUM_LAYERS][n:]])
                for p in task_metrics(shared,name,rows,DIMS):profiles.append(dict(depth=depth,mode='full_corpus',**p))
        scores=[p['score'] for p in profiles if p['depth']!=NUM_LAYERS]
        results[name]=dict(n=n,profiles=profiles,score=float(np.mean(scores)))
        if TASKS[name]=='retrieval':full.extend(p['mrr'] for p in profiles if p['depth']==NUM_LAYERS and p['dim']==DIMS[-1])
    return dict(selection_score=float(np.mean([r['score'] for r in results.values()])),full_score=float(np.mean(full)),by_source=results)


## Stage 3 — train all transformer exits against the frozen selected backbone


In [ ]:
seed_everything(CFG['seed']+100)
bank=ExitBank().to(DEVICE)
bank,EXIT_SELECTED=train_stage(bank,'exits',exit_loss,validate_exits)
assert backbone_digest()==BACKBONE_SHA,'Frozen backbone changed during exit training'
print('Parameters: backbone',sum(p.numel() for p in backbone.parameters()),
      'all exit heads',sum(p.numel() for p in bank.parameters()))


## Learning curves for all three stages


In [ ]:
fig,axes=plt.subplots(3,2,figsize=(13,11))
for i,stage in enumerate(PHASES):
    records={}
    for line in (ROOT/f'{stage}-metrics.jsonl').read_text().splitlines():
        r=json.loads(line);records[r['step']]=r
    rows=[records[k] for k in sorted(records)]
    for name in corpus:
        subset=[r for r in rows if r['source']==name]
        if subset:axes[i,0].plot([r['step'] for r in subset],[r['loss'] for r in subset],alpha=.6,label=name)
        points=[r for r in rows if 'validation' in r]
        axes[i,1].plot([r['step'] for r in points],[r['validation']['by_source'][name]['score'] for r in points],label=name)
    selected=json.loads((ROOT/f'{stage}-selected.json').read_text())['step']
    axes[i,0].set(title=f'{stage}: task losses (different objectives)',xlabel='Update')
    axes[i,1].set(title=f'{stage}: task validation scores',xlabel='Update')
    axes[i,1].axvline(selected,ls='--',color='black');axes[i,0].legend();axes[i,1].legend()
fig.tight_layout();fig.savefig(ROOT/'evaluations'/'training-curves.png',dpi=150);plt.show()


## Independent retrieval and social evaluation


In [ ]:
@torch.inference_mode()
def benchmark_exits():
    results=[]
    for name,b in load_benchmarks().items():
        cids=list(b['corpus']);qids=[q for q in b['queries'] if b['relevant'].get(q)]
        queries=encode_profiles([b['queries'][q] for q in qids],bank)
        docs=encode_profiles([b['corpus'][p] for p in cids],bank)
        for depth in DEPTHS+[NUM_LAYERS]:
            for dim in DIMS:
                for mode,pdepth in [('same_exit',depth),('full_corpus',NUM_LAYERS)]:
                    q=F.normalize(queries[depth][:,:dim],dim=-1);p=F.normalize(docs[pdepth][:,:dim],dim=-1)
                    metrics=retrieval_metrics(q,p,b['relevant'],qids,cids)
                    results.append(dict(dataset=name,depth=depth,dim=dim,mode=mode,**metrics))
                    print(name,depth,dim,mode,'NDCG@10',round(metrics['ndcg@10'],4))
    return results
RETRIEVAL_RESULTS=benchmark_exits()
write_json(ROOT/'evaluations'/'retrieval.json',RETRIEVAL_RESULTS)
# Twitter URL remains an untouched external test, even when its original training split is supplied.
texts=[r[k] for k in ('sentence1','sentence2') for r in TWITTER_EVAL]
vectors=encode_profiles(texts,bank);n=len(TWITTER_EVAL);labels=[r['label'] for r in TWITTER_EVAL]
TWITTER_RESULTS=[]
for depth,z in vectors.items():
    for dim in DIMS:
        a=F.normalize(z[:n,:dim],dim=-1);b=F.normalize(z[n:,:dim],dim=-1);scores=(a*b).sum(-1).numpy()
        TWITTER_RESULTS.append(dict(depth=depth,dim=dim,auc=float(roc_auc_score(labels,scores)),
                                   average_precision=float(average_precision_score(labels,scores))))
write_json(ROOT/'evaluations'/'twitter-url-test.json',dict(n=n,results=TWITTER_RESULTS))
print('Twitter URL:',TWITTER_RESULTS)
del vectors;gc.collect()


## Export one backbone and the selected exits, then reload
The inference bundle excludes GTE, optimizer states, and training-only classification/alignment heads.
Checkpoint files retain those heads for research resumption. Inference consists of the code above plus the
backbone, tokenizer, and exit-head safetensors. ONNX is deferred. Prefix vectors must be sliced then normalized.


In [ ]:
EXPORT=ROOT/'exports'/'social-encoder'
if not EXPORT.exists():
    tmp=ROOT/'exports'/('.uploading-'+uuid.uuid4().hex);tmp.mkdir(parents=True)
    backbone.save_pretrained(tmp/'backbone',safe_serialization=True);tokenizer.save_pretrained(tmp/'tokenizer')
    save_file({k:v.detach().cpu().contiguous() for k,v in bank.state_dict().items()},str(tmp/'exits.safetensors'))
    write_json(tmp/'encoder.json',dict(contract=contract,backbone_sha=BACKBONE_SHA,exit_spec=bank.spec,
        selected=dict(backbone=BACKBONE_SELECTED,prefix=PREFIX_SELECTED,exits=EXIT_SELECTED),
        depths=DEPTHS,dimensions=DIMS,max_length=CFG['short_length'],pooling='masked_mean_including_special_tokens',
        prefix_normalization='slice_then_L2_normalize',full_depth_skips_exit_heads=True))
    shutil.copytree(NOTICE_DIR,tmp/'notices');write_json(tmp/'LICENSE_PROVENANCE.json',LICENSE_RECORD)
    write_json(tmp/'COMMITTED.json',dict(contract=CONTRACT_ID,checksums={str(p.relative_to(tmp)):file_sha(p) for p in tmp.rglob('*') if p.is_file()}))
    os.replace(tmp,EXPORT)
manifest=json.loads((EXPORT/'COMMITTED.json').read_text());assert manifest['contract']==CONTRACT_ID
assert all(file_sha(EXPORT/p)==sha for p,sha in manifest['checksums'].items())
# Exercise backbone/tokenizer AND heads from disk, then restore the in-memory objects.
probe=['forgot my password lol','The library lends books.']
before=encode_profiles(probe,bank)
original_backbone,original_tokenizer=backbone,tokenizer
try:
    backbone=AutoModel.from_pretrained(EXPORT/'backbone',attn_implementation='sdpa').to(DEVICE).eval().requires_grad_(False)
    tokenizer=AutoTokenizer.from_pretrained(EXPORT/'tokenizer')
    loaded=ExitBank().to(DEVICE);loaded.load_state_dict(load_file(str(EXPORT/'exits.safetensors')))
    after=encode_profiles(probe,loaded)
    for depth in before:torch.testing.assert_close(before[depth],after[depth],atol=1e-5,rtol=1e-4)
    del loaded
finally:backbone,tokenizer=original_backbone,original_tokenizer
print('Verified export and reload:',EXPORT)
gc.collect();torch.cuda.empty_cache()


## Final sanity checks at every exit and dimension
These authored probes supplement the held-out retrieval, topic, emotion and paraphrase evaluations.
Numerical/padding failures stop execution. Semantic results are reported without inventing a pass threshold.


In [ ]:
@torch.inference_mode()
def final_sanity():
    bank.eval();texts=[case[i] for i in range(3) for case in SANITY_CASES]
    vectors=encode_profiles(texts,bank,batch_size=4)
    repeated=encode_profiles([SANITY_CASES[0][0]],bank)
    again=encode_profiles([SANITY_CASES[0][0]],bank)
    padded=encode_profiles([SANITY_CASES[0][0],'The library has many books and visitors. '*30],bank)
    reports={}
    for depth,raw in vectors.items():
        report=sanity_scores(raw,SANITY_CASES,DIMS)
        repeat=torch.allclose(repeated[depth],again[depth],atol=1e-5,rtol=1e-4)
        padding={str(d):float(F.cosine_similarity(repeated[depth][:,:d],padded[depth][:1,:d]).item()) for d in DIMS}
        report.update(repeatable=repeat,padding_cosines=padding)
        report['plumbing_passed']=bool(report['finite'] and report['nonzero'] and repeat and min(padding.values())>.999)
        reports[str(depth)]=report
        for d,p in report['profiles'].items():
            print('depth',depth,'dim',d,'hit@1',round(p['hit_at_1'],3),'margin',round(p['mean_margin'],3),
                  'plumbing',report['plumbing_passed'])
    write_json(ROOT/'evaluations'/'final-sanity.json',reports)
    assert all(r['plumbing_passed'] for r in reports.values()),'Inspect saved sanity report before using the export'
    assert backbone_digest()==BACKBONE_SHA
    return reports
FINAL_SANITY=final_sanity()
write_json(ROOT/'RUN_COMPLETE.json',dict(contract=CONTRACT_ID,export=str(EXPORT),
    selected=dict(backbone=BACKBONE_SELECTED,prefix=PREFIX_SELECTED,exits=EXIT_SELECTED),
    twitter_training=bool(TWITTER_TRAIN_FILE),sanity_passed=True))
print('All stages, evaluations, export and reload checks completed. Drive:',ROOT)


## Release the Colab runtime
Run All reaches this cell only after the preceding work completes successfully. This disconnects the runtime
and releases its resources; it does not delete the Drive artifacts. Do not execute it while debugging earlier cells.


In [ ]:
from google.colab import runtime
runtime.unassign()
